In [ ]:
import arcpy
import os
import pandas as pd
import time
import sys

# ==================== 参数配置（绝对路径补全） ====================
# 1. 高程栅格的完整磁盘路径 (例如 C:\Users\...\Data.gdb\AOIs_Evelation_DEM 或 .tif 路径)
dem_raster = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\temp.gdb\AOIs_Evelation_DEM" 

# 2. 50m 缓冲区图层的完整磁盘路径
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Buffer"

# 3. 结果输出路径 (保持你原有的配置)
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\Elevation_Stats_Final.csv"

# ==================== 环境初始化 ====================
arcpy.env.overwriteOutput = True

# 检查并切出 Spatial Analyst 扩展模块许可
if arcpy.CheckExtension("Spatial") == "Available":
    arcpy.CheckOutExtension("Spatial")
else:
    print("错误：无法获取 Spatial Analyst 许可，脚本终止。")
    sys.exit()

# 自动获取 OID 字段名
try:
    desc = arcpy.Describe(buffer_layer)
    oid_fieldname = desc.OIDFieldName
except Exception as e:
    print(f"路径错误：无法找到 Buffer 图层，请检查路径是否正确。错误信息: {e}")
    sys.exit()

print(f"--- 脚本启动：VS Code 独立运行模式 ---")
print(f"检测到 OID 字段: {oid_fieldname}")

# ==================== 初始化监控变量 ====================
total_points = int(arcpy.management.GetCount(buffer_layer)[0])
results = []
start_time = time.time()
processed_count = 0
consecutive_errors = 0

# ==================== 执行核心逻辑 ====================
with arcpy.da.SearchCursor(buffer_layer, ["SHAPE@", oid_fieldname]) as cursor:
    for row in cursor:
        buffer_geom = row[0]
        oid_val = row[1]
        
        try:
            # 1. 动态限制分析范围（提速关键，防止扫描整个 DEM）
            arcpy.env.extent = buffer_geom.extent
            
            # 2. 提取当前 Buffer 区域的栅格
            temp_extract = arcpy.sa.ExtractByMask(dem_raster, buffer_geom)
            
            # 3. 获取统计值
            # 注意：在 VS Code 中，.getOutput(0) 是必须的
            v_min = float(arcpy.management.GetRasterProperties(temp_extract, "MINIMUM").getOutput(0))
            v_max = float(arcpy.management.GetRasterProperties(temp_extract, "MAXIMUM").getOutput(0))
            v_mean = float(arcpy.management.GetRasterProperties(temp_extract, "MEAN").getOutput(0))
            
            results.append([oid_val, round(v_min, 2), round(v_max, 2), round(v_mean, 2)])
            
            processed_count += 1
            consecutive_errors = 0 
            
            # --- 进度监控输出（VS Code 终端可见） ---
            if processed_count % 10 == 0 or processed_count == total_points:
                elapsed = time.time() - start_time
                speed = processed_count / elapsed
                percent_done = (processed_count / total_points) * 100
                remaining = total_points - processed_count
                eta_min = (remaining / speed / 60) if speed > 0 else 0
                
                status_msg = (f"进度: {percent_done:.1f}% ({processed_count}/{total_points}) | "
                              f"速度: {speed:.2f} 点/秒 | 预计剩余: {eta_min:.1f} 分钟")
                print(status_msg)

            # 清理内存中的临时栅格
            arcpy.management.Delete(temp_extract)

        except Exception as e:
            consecutive_errors += 1
            err_log = f"!!! 点 OID {oid_val} 提取失败: {e} (连续失败: {consecutive_errors})"
            print(err_log)
            
            if consecutive_errors >= 3:
                print("\n[熔断] 连续3次失败。程序停止，请检查栅格与矢量的空间位置是否重合。")
                break 

# ==================== 写入结果 ====================
if results:
    print(f"正在保存数据至: {output_csv}")
    df = pd.DataFrame(results, columns=["OID", "Min_Elevation", "Max_Elevation", "Mean_Elevation"])
    df.to_csv(output_csv, index=False)
    
    total_time = (time.time() - start_time) / 60
    print(f"--- 任务完成 --- \n总计成功: {len(df)} \n总耗时: {total_time:.2f} 分钟")
else:
    print("错误：未采集到任何有效数据。")

# 归还许可
arcpy.CheckInExtension("Spatial")

--- 脚本启动：VS Code 独立运行模式 ---
检测到 OID 字段: OBJECTID
进度: 5.0% (10/202) | 速度: 0.20 点/秒 | 预计剩余: 15.7 分钟
